# Multi-Narrowband Beamformer Comparison Example

This notebook runs a shared passive-sonar scenario and compares multiple narrowband
beamformer frequency-band configurations using the same propagation model,
detection pipeline, and target truth.


## Simulation Parameters

This section defines reproducibility and timing for the full run:
random seed, simulation duration, step size, start time, and total timesteps.


In [ ]:
from datetime import datetime, timedelta

import numpy as np

seed = 1
np.random.seed(seed)

sim_length_s = 900
sim_rate_s = 5.0
start_time = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
time_interval = timedelta(seconds=sim_rate_s)
num_steps = int(sim_length_s / sim_rate_s)

total_duration_s = num_steps * time_interval.total_seconds()
print(f"Total simulation duration: {total_duration_s} seconds")

## Platform Setup and Generation

Here we define the ownship trajectory and towed-array geometry.

The host platform follows randomized multi-leg maneuvers generated from
a straight/turn process over the full simulation duration.


In [ ]:
from stonesoup.models.transition.linear import (
    CombinedLinearGaussianTransitionModel,
    ConstantVelocity,
    KnownTurnRate,
)
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState

from nereus.platform import TowedArrayPlatform


def generate_random_maneuvers(
    total_duration_s,
    timestep_s,
    min_leg_duration_s=60,
    max_leg_duration_s=300,
    maneuver_prob=0.5,
):
    """Generate random straight and turning maneuver segments."""
    maneuver_models = []
    maneuver_durations = []
    maneuver_descriptions = []

    current_time = 0.0
    while current_time < total_duration_s - min_leg_duration_s:
        leg_duration = np.random.uniform(min_leg_duration_s, max_leg_duration_s)
        leg_duration = round(leg_duration / timestep_s) * timestep_s
        leg_duration = min(leg_duration, total_duration_s - current_time)

        if np.random.rand() > maneuver_prob:
            model = CombinedLinearGaussianTransitionModel(
                [ConstantVelocity(0.0), ConstantVelocity(0.0), ConstantVelocity(0.0)]
            )
            description = f"Straight ({leg_duration:.0f}s)"
        else:
            turn_angle_deg = np.random.uniform(-90, 90)
            turn_rate_deg_per_s = 1.0

            turn_duration_s = abs(turn_angle_deg) / turn_rate_deg_per_s
            turn_duration_s = round(turn_duration_s / timestep_s) * timestep_s
            leg_duration = min(turn_duration_s, total_duration_s - current_time)

            turn_rate_rad = np.deg2rad(np.sign(turn_angle_deg) * turn_rate_deg_per_s)
            planar_turn = KnownTurnRate(
                turn_rate=turn_rate_rad,
                turn_noise_diff_coeffs=np.array([0.0, 0.0]),
            )
            model = CombinedLinearGaussianTransitionModel(
                [planar_turn, ConstantVelocity(0.0)]
            )

            direction = "Left" if turn_angle_deg > 0 else "Right"
            description = f"{direction} Turn {abs(turn_angle_deg):.0f}° ({leg_duration:.0f}s)"

        maneuver_models.append(model)
        maneuver_durations.append(timedelta(seconds=leg_duration))
        maneuver_descriptions.append(description)
        current_time += leg_duration

    return maneuver_models, maneuver_durations, maneuver_descriptions


platform_start_vector = np.array([5000.0, 0.0, 1000.0, 5.0, -5.0, 0.0])
platform_position_mapping = [0, 2, 4]
platform_velocity_mapping = [1, 3, 5]

num_sensors = 50
tow_cable_length_m = 100.0
sensor_spacing_m = 0.5
array_depth_m = -50.0

maneuver_models, maneuver_durations, maneuver_descriptions = generate_random_maneuvers(
    total_duration_s=total_duration_s,
    timestep_s=time_interval.total_seconds(),
    min_leg_duration_s=60,
    max_leg_duration_s=300,
    maneuver_prob=0.25,
)

initial_state = GroundTruthState(platform_start_vector, timestamp=start_time)
platform = TowedArrayPlatform(
    states=[initial_state],
    position_mapping=platform_position_mapping,
    velocity_mapping=platform_velocity_mapping,
    transition_models=maneuver_models,
    transition_times=maneuver_durations,
    num_sensors=num_sensors,
    cable_length_m=tow_cable_length_m,
    sensor_spacing_m=sensor_spacing_m,
    array_depth_m=array_depth_m,
)

for i in range(1, num_steps):
    platform.move(start_time + i * time_interval)


## Ground Truth Setup and Generation

Target kinematics and source metadata are generated here.

For each target, the notebook propagates motion over all timesteps and computes
the relative-bearing truth sequence with respect to the array reference position.

A world-view plot is shown to validate the geometry before beamforming.


In [ ]:
from nereus.plotter import plot_world

target_start_vectors = [
    np.array([4500.0, 5.0, 2000.0, 0.0, -5.0, 0.0]),
    np.array([1000.0, 5.0, 3000.0, 3.0, -5.0, 0.0]),
]
target_transition_models = [
    CombinedLinearGaussianTransitionModel(
        [ConstantVelocity(0.0), ConstantVelocity(0.0), ConstantVelocity(0.0)]
    ),
    CombinedLinearGaussianTransitionModel(
        [ConstantVelocity(0.0), ConstantVelocity(0.0), ConstantVelocity(0.0)]
    ),
]

target_position_mapping = [0, 2, 4]
target_velocity_mapping = [1, 3, 5]

target_amplitudes_upa = [10 ** (np.array([90.0]) / 20), 10 ** (np.array([100.0]) / 20)]
target_frequencies_hz = [[75.0], [100.0]]
target_phases_rad = [[0.0], [0.0]]
target_tonal_bandwidth_hz = [2.0, 2.0]
target_noise_amplitude_upa = [10 ** (70 / 20), 10 ** (60 / 20)]
target_noise_spectral_exponents = [-1.0, -1.0]

target_ground_truths = []
relative_bearing_ground_truths = []

for (
    target_start_vector,
    target_transition_model,
    amplitudes_upa,
    frequencies_hz,
    phases_rad,
    tonal_bandwidth_hz,
    noise_amplitude_upa,
    noise_spectral_exponent,
) in zip(
    target_start_vectors,
    target_transition_models,
    target_amplitudes_upa,
    target_frequencies_hz,
    target_phases_rad,
    target_tonal_bandwidth_hz,
    target_noise_amplitude_upa,
    target_noise_spectral_exponents, strict=False,
):
    target_metadata = {
        "amplitudes_upa": amplitudes_upa,
        "frequencies_hz": frequencies_hz,
        "phases_rad": phases_rad,
        "position_mapping": target_position_mapping,
        "velocity_mapping": target_velocity_mapping,
        "tonal_bandwidth_hz": tonal_bandwidth_hz,
        "noise_amplitude_upa": noise_amplitude_upa,
        "noise_spectral_exponent": noise_spectral_exponent,
    }

    target_states = [GroundTruthState(target_start_vector, timestamp=start_time, metadata=target_metadata)]
    for i in range(1, num_steps):
        new_time = start_time + i * time_interval
        interval_now = new_time - target_states[-1].timestamp
        new_state_vector = target_transition_model.function(
            target_states[-1],
            noise=False,
            time_interval=interval_now,
        )
        target_states.append(
            GroundTruthState(
                new_state_vector,
                timestamp=new_time,
                metadata=target_states[-1].metadata,
            )
        )

    target_ground_truth = GroundTruthPath(target_states)
    target_ground_truths.append(target_ground_truth)

    bearing_states = []
    for target_state in target_ground_truth.states:
        platform_state = platform.get_platform_state_at(target_state.timestamp)
        ref_sensor_position = np.mean(platform_state.array.state_vector, axis=1)
        target_position = np.array([target_state.state_vector[0], target_state.state_vector[2]])
        relative_position = target_position - ref_sensor_position[:2]
        bearing_rad = np.arctan2(relative_position[1], relative_position[0])
        bearing_states.append(GroundTruthState(state_vector=np.array([bearing_rad]), timestamp=target_state.timestamp))

    relative_bearing_ground_truths.append(GroundTruthPath(bearing_states))

fig_world = plot_world(truths=target_ground_truths, platform=platform)
fig_world.update_layout(title="World Picture: Target and Platform Trajectories")
fig_world.show()


## Propagation Model

This section configures the acoustic environment used by RTRS propagation:
sound-speed profile, bathymetry, and angular/range sampling controls.


In [ ]:
from nereus.models.environment import FlatBathymetry, Linear
from nereus.models.propagation import rtrsAcousticPropagationModel

ssp = Linear(surface_speed=1500.0, gradient=0.2)
bathymetry = FlatBathymetry(depth=-150.0)
prop_step_m = 20.0
prop_azimuth_search_width = 2.0
prop_azimuth_resolution = 0.5
prop_elevation_range = (-25.0, 25.0)
prop_elevation_resolution = 1.0

prop_model = rtrsAcousticPropagationModel(
    ssp=ssp,
    bathymetry=bathymetry,
    step_m=prop_step_m,
    azimuth_search_width=prop_azimuth_search_width,
    azimuth_resolution=prop_azimuth_resolution,
    elevation_range=prop_elevation_range,
    elevation_resolution=prop_elevation_resolution,
)


## Signal Model

Here we define target and ambient signals.

Each target gets a broadband ship signal model with per-target metadata,
while ambient coloured noise is added at the array.


In [ ]:
from nereus.signal.ambient import ColouredNoise
from nereus.signal.anthropogenic import BroadbandShipSignal

sampling_rate_hz = 500.0
frame_len = 500
hop_factor = 2
fade_in_ms = 1000.0

ambient_noise_amplitude_upa = 10 ** (50.0 / 20)
ambient_noise_spectral_exponent = -1

ambient_noise_model = ColouredNoise(
    amplitude_upa=ambient_noise_amplitude_upa,
    spectral_exponent=ambient_noise_spectral_exponent,
    duration_s=time_interval.total_seconds(),
    sampling_rate_hz=sampling_rate_hz,
)

signal_models = []
for target_ground_truth in target_ground_truths:
    target_metadata = next(iter(target_ground_truth)).metadata
    signal_models.append(
        BroadbandShipSignal(
            duration_s=total_duration_s,
            sampling_rate_hz=sampling_rate_hz,
            frame_len=frame_len,
            hop_factor=hop_factor,
            tonal_bandwidth_hz=target_metadata["tonal_bandwidth_hz"],
            noise_amplitude_upa=target_metadata["noise_amplitude_upa"],
            noise_spectral_exponent=target_metadata["noise_spectral_exponent"],
            noise_freq_range_hz=(0.0, sampling_rate_hz / 2),
            tonal_noise_is_constant=True,
            noise_is_constant=True,
        )
    )


## Beamformer

This section sets beamformer parameters and builds multiple beamformer + steering
pairs for direct frequency-band comparison.


In [ ]:
from scipy.signal import get_window

from nereus.sigproc import (
    DelayAndSumBeamformer,
    MinimumVarianceDistortionlessResponseBeamformer,
    SteeringCalculator,
)

steering_azimuths_rad = np.linspace(-np.pi, np.pi, 101)

# Beamformer configuration tuples: (type, shading, domain, fmin_hz, fmax_hz)
beamformer_configs = [
    ("MVDR", None, "broadband_power", 70.0, 105.0),
    ("MVDR", None, "broadband_power", 70.0, 80.0),
    ("MVDR", None, "broadband_power", 95.0, 105.0),
]

beamformers = []
steering_calculators = []
beamformer_labels = []

for beamformer_type, beamformer_shading, beamformer_domain, fmin_hz, fmax_hz in beamformer_configs:
    shading = None
    if beamformer_shading is not None:
        shading = get_window(beamformer_shading, platform.num_sensors)

    if beamformer_type == "DAS":
        if beamformer_domain == "broadband_power":
            beamformer = DelayAndSumBeamformer(
                domain=beamformer_domain,
                sampling_rate_hz=sampling_rate_hz,
                fmin=fmin_hz,
                fmax=fmax_hz,
            )
        else:
            beamformer = DelayAndSumBeamformer(
                sampling_rate_hz=sampling_rate_hz,
                shading=shading,
                domain=beamformer_domain,
            )
    elif beamformer_type == "MVDR":
        beamformer = MinimumVarianceDistortionlessResponseBeamformer(
            sampling_rate_hz=sampling_rate_hz,
            fmin=fmin_hz,
            fmax=fmax_hz,
        )
    else:
        raise ValueError(f"Unknown beamformer type: {beamformer_type}")

    steering_calculator = SteeringCalculator(
        ssp=ssp,
        steering_azimuths_rad=steering_azimuths_rad,
    )

    beamformers.append(beamformer)
    steering_calculators.append(steering_calculator)
    beamformer_labels.append((beamformer_type, fmin_hz, fmax_hz))


## Detector Pipeline Setup

This section builds one simulator+detector pipeline per beamformer configuration.

All pipelines share the same scenario and detector chain so output differences
come from beamformer band selection.


In [ ]:
from nereus.detector import CACFARDetector, PassiveSonarDetector, PeakDetector
from nereus.simulator import BroadbandPassiveSonarArraySimulator

cfar_num_guard_cells = 2
cfar_num_training_cells = 10
cfar_threshold_factor = 2.1
peak_distance = 3


def make_detector(simulator, steering_azimuths_rad):
    cfar_detector = CACFARDetector(
        num_guard_cells=cfar_num_guard_cells,
        num_training_cells=cfar_num_training_cells,
        threshold_factor=cfar_threshold_factor,
    )
    peak_detector = PeakDetector(distance=peak_distance)
    return PassiveSonarDetector(
        detection_chain=[cfar_detector, peak_detector],
        sensor_data_gen=simulator.sensor_data_gen(),
        steering_azimuths_rad=steering_azimuths_rad,
    )

simulators = []
detectors = []
for beamformer, steering_calculator in zip(beamformers, steering_calculators, strict=False):
    simulator = BroadbandPassiveSonarArraySimulator(
        platform=platform,
        propagation_model=prop_model,
        signal_models=signal_models,
        noise_model=ambient_noise_model,
        beamformer=beamformer,
        steering_calculator=steering_calculator,
        ground_truth_paths=target_ground_truths,
        fade_in_ms=fade_in_ms,
    )
    simulators.append(simulator)
    detectors.append(make_detector(simulator, steering_azimuths_rad))


## Run Detection on Simulated Data

This cell executes all detector pipelines and stores detections plus SNR maps
for each beamformer configuration.


In [ ]:
all_detections_per_bf = []
snr_maps = []
detections_flat_per_bf = []

for detector, beamformer_label in zip(detectors, beamformer_labels, strict=False):
    beamformer_type, fmin_hz, fmax_hz = beamformer_label
    all_detections = list(detector.detections_gen(progress_bar=True, total_timesteps=num_steps))
    snr_map = detector.snr_history
    detections_flat = [d for _, detection_set in all_detections for d in detection_set]

    all_detections_per_bf.append(all_detections)
    snr_maps.append(snr_map)
    detections_flat_per_bf.append(detections_flat)

    print(
        f"{beamformer_type} ({fmin_hz:.0f}-{fmax_hz:.0f} Hz): "
        f"{len(detections_flat)} detections"
    )

timesteps = [start_time + i * time_interval for i in range(num_steps)]

## Results: Multi-Narrowband Beamformer Comparison

The first set of figures shows, per beamformer configuration:
- raw SNR map
- SNR map with truth and detections

Then a shared-scale side-by-side comparison is shown, followed by
mid-scenario bearing-cut overlays.


In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from nereus.plotter import plot_btr

# Per-configuration raw vs overlay BTR
for snr_map, detections_flat, beamformer_label in zip(
    snr_maps,
    detections_flat_per_bf,
    beamformer_labels, strict=False,
):
    beamformer_type, fmin_hz, fmax_hz = beamformer_label
    map_rows = min(len(timesteps), snr_map.shape[0])
    timesteps_map = np.array(timesteps[:map_rows])
    snr_map_plot = snr_map[:map_rows]

    fig_btr = make_subplots(
        rows=1,
        cols=2,
        shared_yaxes=True,
        horizontal_spacing=0.08,
        subplot_titles=(
            f"Raw SNR f=({fmin_hz:.0f}-{fmax_hz:.0f} Hz)",
            "SNR, Detections, and Ground Truth",
        ),
    )

    plot_btr(
        data=snr_map_plot,
        timesteps=timesteps_map,
        steering_azimuths=np.rad2deg(steering_azimuths_rad),
        fig=fig_btr,
        row=1,
        col=1,
    )
    plot_btr(
        data=snr_map_plot,
        detections=detections_flat,
        truths=relative_bearing_ground_truths,
        timesteps=timesteps_map,
        steering_azimuths=np.rad2deg(steering_azimuths_rad),
        fig=fig_btr,
        row=1,
        col=2,
    )

    heatmap_count = 0
    for trace in fig_btr.data:
        if trace.type == "heatmap":
            trace.showscale = heatmap_count == 0
            heatmap_count += 1

    fig_btr.update_layout(
        width=1200,
        height=520,
        title=f"Beamformer Output: {beamformer_type} ({fmin_hz:.0f}-{fmax_hz:.0f} Hz)",
    )
    fig_btr.update_yaxes(title_text="Time (HH:MM)", row=1, col=1)
    fig_btr.update_yaxes(title_text="", row=1, col=2)
    fig_btr.show()

# Shared-scale cross-configuration BTR comparison
comparison_titles = [
    f"{beamformer_type} f=({fmin_hz:.0f}-{fmax_hz:.0f} Hz)"
    for beamformer_type, fmin_hz, fmax_hz in beamformer_labels
]

fig_compare = make_subplots(
    rows=1,
    cols=len(beamformer_labels),
    shared_yaxes=True,
    horizontal_spacing=0.04,
    subplot_titles=comparison_titles,
)

for i, snr_map in enumerate(snr_maps, start=1):
    map_rows = min(len(timesteps), snr_map.shape[0])
    timesteps_map = np.array(timesteps[:map_rows])
    snr_map_plot = snr_map[:map_rows]
    plot_btr(
        data=snr_map_plot,
        timesteps=timesteps_map,
        steering_azimuths=np.rad2deg(steering_azimuths_rad),
        fig=fig_compare,
        row=1,
        col=i,
    )

vmin = 0.0
vmax = 40.0
heatmap_count = 0
for trace in fig_compare.data:
    if trace.type == "heatmap":
        trace.zmin = vmin
        trace.zmax = vmax
        trace.showscale = heatmap_count == 0
        heatmap_count += 1

fig_compare.update_layout(
    width=1400,
    height=520,
    title="SNR Comparison Across Beamformer Configurations",
    showlegend=False,
)
fig_compare.update_yaxes(title_text="Time (HH:MM)", row=1, col=1)
fig_compare.show()

# Mid-scenario bearing cuts
for all_detections, snr_map, beamformer_label in zip(
    all_detections_per_bf,
    snr_maps,
    beamformer_labels,
):
    beamformer_type, fmin_hz, fmax_hz = beamformer_label
    detections_by_time = [detection_set for _, detection_set in all_detections]

    middle_timestep_idx = min(num_steps // 4, snr_map.shape[0] - 1)
    middle_snr = snr_map[middle_timestep_idx, :]
    middle_time = timesteps[middle_timestep_idx]
    steering_azimuths_deg = np.rad2deg(steering_azimuths_rad)

    y_span = max(float(np.ptp(middle_snr)), 1.0)
    y_pad = 0.05 * y_span
    y_min = float(np.min(middle_snr) - y_pad)
    y_max = float(np.max(middle_snr) + y_pad)

    fig_middle = go.Figure()
    fig_middle.add_trace(
        go.Scatter(
            x=steering_azimuths_deg,
            y=middle_snr,
            mode="lines",
            line=dict(width=2, color="steelblue"),
            name="SNR",
        )
    )

    colorway = px.colors.qualitative.Plotly
    for target_idx, gt_path in enumerate(relative_bearing_ground_truths):
        gt_bearing_deg = float(np.rad2deg(gt_path.states[middle_timestep_idx].state_vector[0]))
        fig_middle.add_trace(
            go.Scatter(
                x=[gt_bearing_deg, gt_bearing_deg],
                y=[y_min, y_max],
                mode="lines",
                line=dict(color=colorway[target_idx % len(colorway)], dash="dash", width=2),
                name=f"Target {target_idx + 1} GT ({gt_bearing_deg:.1f}°)",
            )
        )

    middle_detections = list(detections_by_time[middle_timestep_idx])
    for det_idx, det in enumerate(middle_detections):
        det_bearing_deg = float(np.rad2deg(det.state_vector[0]))
        fig_middle.add_trace(
            go.Scatter(
                x=[det_bearing_deg, det_bearing_deg],
                y=[y_min, y_max],
                mode="lines",
                line=dict(color="green", dash="dot", width=1.5),
                name="Detections",
                showlegend=det_idx == 0,
                opacity=0.75,
            )
        )

    fig_middle.update_layout(
        template="plotly_white",
        width=950,
        height=460,
        title=(
            f"SNR Bearing Cut: {beamformer_type} "
            f"({fmin_hz:.0f}-{fmax_hz:.0f} Hz) at {middle_time}"
        ),
        xaxis_title="Bearing (degrees)",
        yaxis_title="SNR (dB)",
    )
    fig_middle.update_yaxes(range=[y_min, y_max])
    fig_middle.show()
